# Ontology: concepts, queries, and overrides

The ontology is the workspace's semantic layer: concepts are the shared
meaning of columns across sources, and the query engine joins, filters, and
aggregates through them so you never hand-write reconciliation SQL. This
notebook uploads a small dataset, works with the concepts the ingest built,
queries through them, and overrides what the profiler detected.

In [ ]:
import os

import numpy as np
import pandas as pd

import rootcause as rc

rc.login(base_url=os.environ.get("ROOTCAUSE_BASE_URL", "https://platform.rootcause.ai"))
ws = rc.workspace("ontology-demo", create=True)

## Concepts arrive with the data

Every upload gets concepts during ingest — profiled, typed, and named:

In [ ]:
rng = np.random.default_rng(29)
n = 240
marketing = rng.normal(50, 12, n)
leads = 2.1 * marketing + rng.normal(0, 8, n)
frame = pd.DataFrame({
    "region": rng.choice(["emea", "apac", "amer"], n),
    "marketing_spend": marketing.round(2),
    "leads": leads.round(1),
    "revenue": (3.4 * leads + rng.normal(0, 15, n)).round(1),
    # a running total with occasional dedup corrections — it dips, so the
    # profiler will rightly refuse to call it monotonic on the data alone
    "cumulative_signups": np.cumsum(rng.integers(1, 9, n)) - np.where(rng.random(n) < 0.06, 4, 0),
})
source = ws.upload(frame, "campaign-weeks")

# ontology analysis runs just after ingest; wait for the concepts to land
import time
onto = ws.ontology
while len(onto.concepts) < frame.shape[1]:
    time.sleep(2)
onto.concepts

`onto[...]` hands back a **concept handle** — resolve once, then every
operation lives on the object. If a name matches more than one concept, the
lookup refuses with the candidates listed (`onto.matching(name)` disambiguates).

In [ ]:
signups = onto["Cumulative Signups"]
signups

## Query through concepts

`query()` selects, filters, groups, and orders through concept names — the
engine compiles it against the underlying sources:

In [ ]:
onto.query(
    select=["Region", "Marketing Spend", "Revenue"],
    where=[("Revenue", ">=", 400)],
    order_by="Revenue",
).to_frame().tail(5)

## Override what the profiler detected

The running total dips whenever duplicates are removed, so on the data alone
the profiler rightly refuses to flag it monotonic. Semantically it can never
go down — and the model should know that. Override it on the handle: fields
are set and **locked**, so future ingests preserve your values while the
detected ones keep shadowing underneath:

In [ ]:
signups.override(monotonically_increasing=True, description="Running total of signups")
signups.locks

Forecasts apply a running clamp to any node whose concept is flagged
monotonically increasing — a cumulative metric can never forecast downward.
Other keywords cover units, value ranges, fill strategies, categories, and the
concept-level role (`suggested_role="target"` sets the default for every
future twin built over the concept). Overriding a field to the value the
profiler already detected is a deliberate no-op — locks exist only where you
actually disagree.

`revert()` hands fields back to the profiler — detected values restored,
locks lifted:

In [ ]:
signups.revert()
signups.locks

From here the concepts feed everything else: twins built over this workspace
inherit the roles and metadata, and [temporal-panel.ipynb](temporal-panel.ipynb)
shows the modelling side.